# chatbot with sqlite checkpoint & tools

In [14]:
from pathlib import Path
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from typing import TypedDict, Annotated
from langgraph.graph import add_messages, StateGraph, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import ToolNode

import os
os.makedirs("logs", exist_ok=True)
import sys
import logging
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(filename='logs/chatbots.log', mode='a')
    ]
)
logger = logging.getLogger(__name__)

from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
MODEL = "gemini-2.5-flash"
TEMPERATURE = 0.1

BASE_DIR = Path.cwd()  
DB_DIR = BASE_DIR / "db/chatbots"
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = str(DB_DIR / "checkpoint.sqlite")

MAX_HISTORY = 3 # keep last N non-system messages
SUMMARIZE_THRESHOLD = 7 # when total messages exceed this, summarize

In [16]:
class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

In [17]:
class ChatBot:
    def __init__(
        self,
        model: str = MODEL,
        temperature: float = TEMPERATURE,
        db_path: str = DB_PATH,
        max_history: int = MAX_HISTORY,
        summarize_threshold: int = SUMMARIZE_THRESHOLD,
    ):
        self.model = model
        self.temperature = temperature
        self.db_path = db_path
        self.max_history = max_history
        self.summarize_threshold = summarize_threshold

        self.tools = self._setup_tools()
        self.summarizer_llm = ChatGoogleGenerativeAI(
            model="gemini-1.5-flash",
            temperature=0.2
        )
        self.llm = self._build_llm()
        self.checkpointer = self._build_checkpointer()
        self.app = self._build_graph()

    def _setup_tools(self):
        try:
            search_tool = TavilySearchResults(max_results=3)
            tools = [search_tool]

            return tools
        except Exception as e:
            logger.error(f"❌ Error setting up tools: {e}")
            raise

    def _build_llm(self) -> ChatGoogleGenerativeAI:
        try:
            llm = ChatGoogleGenerativeAI(
                model=self.model,
                temperature=self.temperature,
            )
            llm = llm.bind_tools(tools=self.tools)
            logger.info(f"✅ Initialized LLM: {self.model} with temperature {self.temperature}")
            return llm
        except Exception as e:
            logger.error(f"❌ Error initializing LLM: {e}")
            raise

    def _build_checkpointer(self) -> SqliteSaver:
        try:
            sqlite_connection = sqlite3.connect(database=self.db_path, check_same_thread=False)
            checkpointer = SqliteSaver(sqlite_connection)
            logger.info(f"✅ Initialized Checkpointer with DB: {self.db_path}")
            return checkpointer
        except Exception as e:
            logger.error(f"❌ Error initializing Checkpointer: {e}")
            raise

    def _build_graph(self):
        graph = StateGraph(ChatState)

        graph.add_node('chatbot', self.chatbot)
        graph.add_node('tool_node', ToolNode(self.tools))

        graph.set_entry_point('chatbot')

        graph.add_conditional_edges(
            'chatbot',
            self.tool_router,
            {
                'tool_node': 'tool_node',
                'end': END
            }
        )
        graph.add_edge('tool_node', 'chatbot')
        
        app = graph.compile(checkpointer=self.checkpointer)

        logger.info("✅ Compiled StateGraph")
        return app
    
    def _maybe_summarize(self, messages):
        """Summarize earlier part of conversation to control context size."""
        if len(messages) < self.summarize_threshold:
            return messages
        # Prevent repeated summarization
        if any(isinstance(m, SystemMessage) and "Conversation summary" in (m.content or "") for m in messages):
            return messages
        try:
            summary_text = self.summarizer_llm.invoke([
                SystemMessage(content="Summarize the earlier conversation briefly focusing on facts & user goals."),
                HumanMessage(content="\n\n".join(
                    f"{m.type.upper()}: {getattr(m,'content','')}" for m in messages[:-self.max_history]
                ))
            ]).content
            summary_msg = SystemMessage(content=f"Conversation summary: {summary_text}")
            kept_tail = messages[-self.max_history:]
            logger.info("🧾 Added conversation summary to reduce token usage.")
            logger.debug(f"🧾 Summary: {summary_text}")
            logger.debug(f"🧾 Kept tail messages: \n{kept_tail}")

            return [summary_msg] + kept_tail
        except Exception as e:
            logger.warning(f"Summary failed, proceeding without summarization: {e}")
            return messages
        
    def _trim_messages(self, messages):
        # Keep any system / summary messages plus last N others
        system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
        non_system = [m for m in messages if m not in system_msgs]
        tail = non_system[-self.max_history:]
        trimmed = system_msgs + tail
        logger.debug(f"🗜️ Trimmed messages: \n{trimmed}")

        return trimmed
    
    def _log_tool_calls(self, msg):
        """Pretty-print tool calls (if any) for debugging."""
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for i, tc in enumerate(msg.tool_calls, 1):
                name = tc.get("name")
                args = tc.get("args")
                logger.debug(f"🛠 Tool call {i}: name={name} args={args}")
        else:
            logger.debug("🛠 No tool calls in this AIMessage.")

    def chatbot(self, state: ChatState):
        messages = state['messages']

        # Summarize if large
        messages = self._maybe_summarize(messages)
        # Trim window
        messages_to_send = self._trim_messages(messages)

        logger.debug(f"➡️ Sending to LLM ({self.model}) total_messages={len(messages)}")
        logger.debug(f"sent_messages={len(messages_to_send)} last_user='{messages[-1].content if messages else ''}'")

        response = self.llm.invoke(messages_to_send)

        self._log_tool_calls(response)

        logger.debug(f"⬅️ LLM response: '{response.content if response else ''}'")

        return {
            "messages": [response]
        }
    
    def tool_router(self, state: ChatState):
        last_message = state['messages'][-1]
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            logger.debug("🔀 Routing: tool calls detected -> tool_node")
            return 'tool_node'
        logger.debug("🔀 Routing: no tool calls -> end")
        return 'end'

In [18]:
def main():
    bot = ChatBot()
    app = bot.app

    config = {"configurable": {
        "thread_id": 1
    }}

    while True:
        user_input = input("You (q to exit): ")
        print(f"You: {user_input}")
        if user_input.lower() in ['q']:
            break
        else:
            result = app.invoke({
                "messages": [HumanMessage(content=user_input)],
            }, config=config)

            logger.debug(f"📝 State after invocation: {result}")

            print(f"Assistant: {result['messages'][-1].content}")

In [19]:
main()

2025-08-27 13:46:57 | INFO     | __main__ | ✅ Initialized LLM: gemini-2.5-flash with temperature 0.1
2025-08-27 13:46:57 | INFO     | __main__ | ✅ Initialized Checkpointer with DB: /Users/hi/jitenStuff/MyGit/Artificial-Intelligence/Tool-Framework-Library/LangGraph/HelloWorld/db/checkpoint.sqlite
2025-08-27 13:46:57 | INFO     | __main__ | ✅ Compiled StateGraph
You: When was SpaceX's last launch?
2025-08-27 13:47:10 | INFO     | __main__ | 🧾 Added conversation summary to reduce token usage.
2025-08-27 13:47:10 | DEBUG    | __main__ | 🧾 Summary: The user, Jiten, introduced themselves and then asked for a one-line description of an AI agent.  Finally, they asked for the date of SpaceX's last launch (the AI's response is incomplete in this provided transcript).
2025-08-27 13:47:10 | DEBUG    | __main__ | 🧾 Kept tail messages: 
[ToolMessage(content='[{"title": "Mission - SpaceX", "url": "https://www.spacex.com/mission/", "content": "On April 8, 2016, the Falcon 9 rocket launched the Dragon 